# Single-sample Gemini Audio Inference (Arabic)

Runs `gemini-2.5-flash` for the missing Arabic sample in `uh-mazing_gemini-2.5-flash_audio_standard-noextra.csv` and prints the output.

In [1]:
import os
from pathlib import Path

import google.generativeai as genai
import pandas as pd
from dotenv import load_dotenv


/opt/anaconda3/envs/text-irr/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/anaconda3/envs/text-irr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/n2/yfmlyvts65xd64jgl_j_5jk80000gn/T/ipykernel_35309/2794715442.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See R

In [2]:
load_dotenv()

MODEL_NAME = 'gemini-2.5-flash'
INPUT_CSV = Path('../outputs/uh-mazing_gemini-2.5-flash_audio_standard-noextra.csv')
AUDIO_DIR = Path('../asr/preprocessed_audio_switchboard')
ROW_INDEX = 65
TARGET_LANG = 'Arabic'
PROMPT = (
    'Translate the speech audio into <LANGUAGE> text. '
    'Only return the answer requested. Do not include any explanation or introductions.'
).replace('<LANGUAGE>', TARGET_LANG)

api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise RuntimeError('Missing GEMINI_API_KEY (or GOOGLE_API_KEY)')

genai.configure(api_key=api_key)
model = genai.GenerativeModel(MODEL_NAME)

df = pd.read_csv(INPUT_CSV)
row = df.loc[ROW_INDEX]
audio_name = str(row['_audio_filename'])
audio_path = AUDIO_DIR / audio_name

print('Row index:', ROW_INDEX)
print('ID:', row['ID'])
print('Audio file:', audio_path)
print('Current AR value:', row.get('AR_gemini-2.5-flash'))

if not audio_path.exists():
    raise FileNotFoundError(f'Audio not found: {audio_path}')


Row index: 65
ID: sw2024_B_4
Audio file: ../asr/preprocessed_audio_switchboard/sw02024_B_4.wav
Current AR value: nan


In [3]:
audio_bytes = audio_path.read_bytes()
response = model.generate_content(
    [
        PROMPT,
        {"mime_type": "audio/wav", "data": audio_bytes},
    ],
    generation_config={"temperature": 0.0},
)

translation = (response.text or "").strip()
print("\nArabic translation:")
print(translation)



Arabic translation:
أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه، أه،

In [4]:
OUTPUT_TXT = Path('../llm-as-a-judge/missing_arabic_row65.txt')
OUTPUT_TXT.write_text(translation + '\n', encoding='utf-8')
print(f'Saved translation to: {OUTPUT_TXT.resolve()}')


Saved translation to: /Users/mariateleki/Desktop/Uh-Mazing/llm-as-a-judge/missing_arabic_row65.txt
